# Topic: Python Port Scanner - TCP socket connection attempts, banner grabbing, and thread pool acceleration
 
## 1. THE CONCEPT OF PORT SCANNING
- **Port scanning**: A reconnaissance technique used to map active services running on a target host.
- **TCP Full Connect Scan**:
  - Establishes a full TCP 3-way handshake (SYN -> SYN-ACK -> ACK).
  - Python's standard `socket.connect((ip, port))` or `socket.connect_ex((ip, port))` performs a full connect scan.
  - **`connect_ex` benefit**: Returns an error integer code (like `0` for success or standard POSIX error codes like `111` / `errno.ECONNREFUSED` for closed/filtered ports) instead of raising exceptions.
- **Banner Grabbing**:
  - Once a port is verified as open, the scanner sends an initial request (or listens) to read metadata strings returned by the server (e.g. `"SSH-2.0-OpenSSH_8.9p1"`). This reveals service names and version numbers, useful for mapping vulnerabilities.
 
## 2. SCAN ACCELERATION & TIMEOUTS
- **Timeout limit**: Setting a low socket timeout (e.g., `1.0` seconds) is critical to prevent scanning threads from hanging indefinitely on filtered ports (ports blocked by firewalls that drop packets silently without returning RST responses).
- **Concurrency**: Using `concurrent.futures.ThreadPoolExecutor` allows scanning multiple ports in parallel, reducing execution times significantly.
 
---


In [ ]:
import socket
import errno
from concurrent.futures import ThreadPoolExecutor

# Target host definition (localhost for safe lab-only scanning)
TARGET_HOST = "127.0.0.1"

# 1. Custom single-port scanner function with Banner Grabbing
def scan_port(host, port, timeout=1.0):
    """Scans a single port. If open, attempts banner grabbing."""
    # Create standard TCP socket
    # AF_INET = IPv4 | SOCK_STREAM = TCP
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout)
    
    try:
        # connect_ex returns 0 if connection succeeded
        result = s.connect_ex((host, port))
        
        if result == 0:
            banner = "Unknown Service"
            try:
                # Send generic request or wait to read initial banner
                # Some protocols (like SSH, FTP) send banners immediately
                banner_bytes = s.recv(1024)
                if banner_bytes:
                    banner = banner_bytes.decode('utf-8', errors='ignore').strip()
            except socket.timeout:
                # No banner received within timeout, but port is open
                pass
            
            s.close()
            return port, True, banner
        else:
            s.close()
            return port, False, ""
    except Exception:
        s.close()
        return port, False, ""

# Test single port scan on a common port (e.g. port 80 or similar)
port_test = 80
port, is_open, banner = scan_port(TARGET_HOST, port_test, timeout=0.5)
print(f"Single Port Scan -> Port: {port} | Open? {is_open} | Banner: {banner}")



In [ ]:
# 2. Concurrent Port Scanner using ThreadPoolExecutor
def run_concurrent_scanner(host, ports, max_workers=50):
    print(f"\n--- Starting Concurrent Scan on {host} ---")
    open_ports = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit scanning tasks
        futures = {executor.submit(scan_port, host, p, 0.5): p for p in ports}
        
        for future in futures:
            port, is_open, banner = future.result()
            if is_open:
                print(f"[+] Port {port:>5} is OPEN | Service Banner: {banner}")
                open_ports.append((port, banner))
                
    print("--- Scan Finished ---")
    return open_ports

# Define safe target list of ports (common development/system ports)
test_ports = [21, 22, 80, 443, 8080, 9000]
run_concurrent_scanner(TARGET_HOST, test_ports)
